<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Project 1: Midterm Project

In this project, you will use the skills and concepts you've learned so far in the course to explore a question that sounds simple: **why do some people plan a route better than others?** You will analyze real data from an online lawn mowing game built by The Pudding for their story [Why some people mow a lawn better than others](https://pudding.cool/2026/06/mow/), using tables, visualizations, sampling, hypothesis testing, and A/B testing.

---

## 🎗️ Assignment Reminders

As you work on this project, consider the following:
- **Initialize Otter.** 🚨 Make sure to run the code cell at the top of this notebook that starts with `# Initialize Otter` to load the auto-grader.
- **Complete the Tasks.** Your Tasks are categorized as auto-graded (📍) and manually graded (📍🔎).
    - For all the auto-graded tasks:
        - Replace the `...` in the provided code cell with your own code.
        - Run the `grader.check` code cell to run some tests on your code.
        - Keep in mind that for homework and project assignments, sometimes there are hidden tests that you will not be able to see the results of, which we use for scoring the correctness of your response. **Passing the auto-grader does not guarantee that your answer is correct.**
    - For all the manually graded tasks:
        - You might need to provide your own response to the provided prompt. Do so by replacing the template text "_Type your answer here, replacing this text._" with your own words.
        - You might need to produce a graphic or something else using code. Do so by replacing the `...` in the code cell to generate the image, table, etc.
        - In either case, [review the rubric](https://community.canvaslms.com/t5/Student-Guide/How-do-I-view-the-rubric-for-my-assignment/ta-p/275) on the associated <a href="https://ccsf.instructure.com" target="_blank">Canvas</a> Assignment page to understand the scoring criteria.
- **Code Sharing.** By submitting this project, you agree that you will not share your code directly with anybody but your partner. You are welcome to discuss questions with others but don't share the answers directly. The experience of solving the problems in this project will prepare you for our exams and potentially for future work in this field. If someone asks you for the answer, resist! Instead, you can demonstrate how you would solve a similar problem or you can focus on a specific part of a task.
- **Support.** You are not alone! Review the [Course Support page](https://ccsf-math-108.github.io/materials-fa26/resources/course-support.html) for how to get support in this course. If you're ever feeling overwhelmed or don't know how to make progress, talk to your instructor or a tutor.
- **Advice.** Develop your answers incrementally. To perform a complicated table manipulation, break it up into steps, perform each step on a different line, give a new name to each result, and check that each intermediate result is what you expect. You can add any additional names or functions you want to the provided cells. Make sure that you are using distinct and meaningful variable names throughout the notebook.
- **Variable Names.** Throughout this assignment and all future ones, please be sure to not re-assign variables throughout the notebook! _For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you will fail tests that you thought you were passing previously!_
- **Re-Submitting.** You may [submit](#🏁-Submit-Your-Assignment-to-Canvas) this assignment as many times as you want before the deadline. Your instructor will score the last version you submit once the deadline has passed.

---

## Set Up Notebook

### Task 00 📍

<!-- BEGIN QUESTION -->

Run the following code cells to set up the notebook.

_You are not responsible for understanding what this code does._

_Points:_ 0

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("project1.ipynb")

In [ ]:
# Configure this Notebook
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

<!-- END QUESTION -->

---

## 🌱 Section 1: The Lawn Mowing Experiment

<img src="./game_screenshot.png" alt="A screenshot of the game in the form of a grid with grass in some blocks and rocks (obstacles) in others. " width="400px">

Imagine a lawn drawn as a grid of squares. Some squares are grass that must be mowed, and some are obstacles. You control a mower that moves one square at a time (up, down, left, or right), and your goal is to mow every grass square. Sounds easy, but you quickly find that the order in which you cover the squares matters: if you paint yourself into a dead end, you have to backtrack over squares you already mowed, and every extra step is wasted effort.

---

### Optimality Score

Computer scientists call a version of this the **coverage path planning** problem, a close cousin of the famous _Traveling Salesman Problem_. A computer can search exhaustively for the shortest possible route on each lawn, called the **optimal** route. A player's score on an attempt is how close they came to that route.

$$\text{optimality} = 100 \times \frac{\text{optimal moves}}{\text{moves the player actually made}}$$

An optimality of 100% means a perfect solution, while lower values mean more wasted moves.

---

### The Data

The game has six lawns. Everyone starts with a **Tutorial** and then plays **Level 1** and **Level 2**. Players who choose to keep going can play the bonus lawns, **Levels 3, 4, and 5**, which are more challenging.

The data you will use in this project comes from real volunteers who played the game online. [The raw data](https://github.com/the-pudding/data/tree/master/mow) contained the full path of every player on every lawn, one row per player per lawn, as a long list of $x$/$y$ positions and timestamps.

To make the data easier to work with, we processed the raw paths into three tables for you. 

Run the following code cell to load them.

In [ ]:
levels = Table.read_table('levels.csv')
attempts = Table.read_table('attempts.csv')
demographics = Table.read_table('demographics.csv')
levels.show()
attempts.show(5)
demographics.show(5)

---

The `levels` table has one row per lawn:

| Column | Description |
|---|---|
| `level` | The name of the lawn (`Tutorial`, `Level 1`, ..., `Level 5`) |
| `grid` | The size of the grid |
| `required_squares` | How many grass squares must be mowed |
| `optimal_moves` | The number of moves in the shortest possible route |
| `stage` | `Tutorial`, `Required`, or `Bonus` |

The `attempts` table has one row per attempt by a player on a lawn:

| Column | Description |
|---|---|
| `user_id` | A 10 character identifier for the player |
| `level` | Which lawn was attempted |
| `platform` | `desktop` (keyboard) or `mobile` (on-screen keypad) |
| `moves` | The number of squares the mower visited in order, counting the starting square and any square visited more than once |
| `seconds` | Total seconds from the start of the attempt to the last move |
| `first_move_seconds` | Seconds the player waited before making their first move |
| `longest_pause_seconds` | The longest gap, in seconds, between two consecutive moves |
| `pauses` | The number of gaps of at least 2 seconds between consecutive moves |

The `demographics` table has one row per player and comes from an optional survey shown after Level 2. Players who skipped a question have the value `No answer`.

| Column | Description |
|---|---|
| `user_id` | The same player identifier used in `attempts` |
| `age` | Age group |
| `style` | How the player describes the way they mow a lawn or vacuum |
| `gaming` | How often the player plays video games |
| `hand` | Left or right handed |
| `optimization` | How often the player's work or studies involve optimization or routing problems |

**Note**: A few attempts include very long idle times (for example, someone who left the game open in a browser tab for hours). We have not removed them.

---

### Task 1.1 📍

How large is this dataset? Using `attempts` and table methods, assign `num_attempts` to the total number of attempts (rows) and `num_players` to the number of **distinct** players.

**Hint**: Grouping a table by a column creates one row for each distinct value in that column.

_Points:_ 2

In [ ]:
num_attempts = ...
num_players = ...

# Leave the following code to print out the information.
print(f'The dataset contains {num_attempts} attempts (rows) and for a total of {num_players} distinct players.')


In [ ]:
grader.check("task_1_1")

---

### Task 1.2 📍

The number of moves in the optimal route lives in `levels`, while each player's actual moves live in `attempts`. To compare them, combine the two tables. Using the `join` method, create a table called `attempts_full` that adds the information from `levels` to every row of `attempts`.

**Hint**: Both tables have a `level` column.

_Points:_ 2

In [ ]:
attempts_full = ...
attempts_full.show(3)

In [ ]:
grader.check("task_1_2")

---

### Task 1.3 📍

Create a new table called `mowing` by adding two columns to `attempts_full`:

* `'optimality'`: the optimality score defined above, on a scale from 0 to 100.
* `'perfect'`: `True` if the player made exactly the optimal number of moves and `False` otherwise.

**Hint**: Column arithmetic and comparisons work element by element on arrays, like `tbl.column('a') / tbl.column('b')` or `tbl.column('a') == tbl.column('b')`.

_Points:_ 3

In [ ]:
mowing = ...
mowing.show(3)

In [ ]:
grader.check("task_1_3")

---

### Task 1.4 📍

Using `mowing` and table methods, create a table called `median_by_level` with one row per lawn that shows the **median** optimality for that lawn.

**Hint**: Select only the columns you need before grouping so the output is easy to read.

_Points:_ 2

In [ ]:
median_by_level = ...
median_by_level

In [ ]:
grader.check("task_1_4")

---

### Task 1.5 📍

In [the article](https://pudding.cool/2026/06/mow/), the authors report that roughly 16% of players solved one of the lawns perfectly, and that about half of players came within five moves of the optimal route. Use your data to find out which lawn those numbers describe.

1. Define a function `percent_within(level, extra_moves)` that returns the **percent** of attempts on `level` where `moves` is less than or equal to `optimal_moves + extra_moves`. Return the result as a number from 0 to 100, not a proportion from 0 to 1. For example, `percent_within('Level 2', 0)` should return the percent of attempts on Level 2 that were perfect (used exactly the optimal number of moves).
2. Use `percent_within` to assign these two variables:
   - `percent_perfect_level2`: the percent of attempts on `'Level 2'` that were perfect.
   - `percent_within5_level2`: the percent of attempts on `'Level 2'` that used at most 5 moves more than optimal.

_Points:_ 3

In [ ]:
# Define the function percent_within
...

# Call the function
percent_perfect_level2 = ...
percent_within5_level2 = ...

# Leave the following code to print out the calculated values.
print(f'The percent of perfect attempts on Level 2 is approximately {percent_perfect_level2:.2f}%.')
print(f'The percent within 5 moves of a perfect attempt is approximately {percent_within5_level2:.2f}%.')


In [ ]:
grader.check("task_1_5")

---

### Task 1.6 📍🔎

<!-- BEGIN QUESTION -->

Compare how players did on a small lawn (Level 1) and the largest lawn (Level 5). Create overlaid histograms of `optimality` for the two lawns, using `level` as the group. Use `np.arange(60, 101, 5)` for the bins.

**Hint**: Use `where` with `are.contained_in(make_array('Level 1', 'Level 5'))` to keep only the two lawns.

_Points:_ 2

In [ ]:
...

# Keep this code to give your image a title
plt.title('Optimality on Level 1 vs. Level 5')
plt.show()

<!-- END QUESTION -->

---

### Task 1.7 📍

After Level 2, players were invited to answer an optional survey, which contained a question about how often the player plays video games. Use those results to explore whether players who play video games more often do better on Level 2.

* Create a table called `level2_survey` by joining the attempts on **Level 2** (from `mowing`) with the `demographics` table.
* Then create a table called `gaming_means` with one row for each answer in the `gaming` column that shows the **average** optimality for that answer.

**Hint**: The two tables share the `user_id` column.

_Points:_ 3

In [ ]:
level2_survey = ...
gaming_means = ...
gaming_means

In [ ]:
grader.check("task_1_7")

---

### Task 1.8 📍🔎

<!-- BEGIN QUESTION -->

In the article, the authors found that the total time a player spends explains very little of how well they do. Check whether this idea shows up in the real data. First, take the first 2,000 attempts on **Level 2** that lasted **less than 300 seconds** (this leaves out the idle attempts) and create a scatter plot of `optimality` (vertical axis) against `seconds` (horizontal axis). Then, in the markdown cell that follows, describe in one or two sentences what you see. Is a player who takes longer reliably a better player?

**Hint**: After filtering with `where`, you can use `take(np.arange(2000))` to keep the first 2,000 rows.

_Points:_ 3

In [ ]:
...

# Keep this code to give your image a title
plt.title('Optimality vs. Time on Level 2')
plt.show()

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

## 🎲 Section 2: How Much Can We Trust a Sample?

The article's authors were able to look at everyone who played their game. Most researchers can't. Usually you only see a **random sample** of a much larger population, and you have to decide how much a sample statistic can be trusted.

To build intuition, we will pretend that every attempt on Level 2 in our data is the entire **population** of interest. Because we can see the whole population, we know the truth (the true proportion of perfect attempts), and we can study how well random samples of different sizes capture it.

Run the following code cell to create a table of the Level 2 attempts.

In [ ]:
level2 = mowing.where('level', 'Level 2')
level2.show(3)

---

### Task 2.1 📍

Compute the proportion (a number between 0 and 1) of attempts in `level2` that were perfect, and assign it to `level2_perfect_prop`. 

This is the **population parameter** we are trying to estimate.

_Points:_ 2

In [ ]:
level2_perfect_prop = ...
level2_perfect_prop

In [ ]:
grader.check("task_2_1")

---

### Task 2.2 📍

Define a function called `sample_perfect_prop` that takes a sample size as its only argument. It should:

* Draw a **simple random sample without replacement** of that many attempts from `level2`.
* Return the proportion of attempts in the sample that were perfect.

Lastly, call your function with a sample size of `100` and assign the result to `one_sample_prop`.

**Hint**: Look up the arguments of the `sample` table method.

_Points:_ 3

In [ ]:
# Define the function
...

# Call the function with a sample size of 100
one_sample_prop = ...
one_sample_prop

In [ ]:
grader.check("task_2_2")

---

### Task 2.3 📍

Now see how sample proportions behave when you repeat the sampling many times. 

Use a `for` loop to draw **2,000** random samples of size **100** and store the 2,000 sample proportions in an array called `sample_props_100`. Then do the same for samples of size **1,000** and store the results in `sample_props_1000`.

**Hint**: Start with an empty array made by `make_array()` and use `np.append` inside the loop. This may take several seconds to run.

_Points:_ 4

In [ ]:
sample_props_100 = ...
sample_props_1000 = ...

...

In [ ]:
grader.check("task_2_3")

---

### Distribution of Sample Proportions

Run the following code to compare the two empirical distributions of sample proportions. The red triangle marks the true population proportion.

In [ ]:
Table().with_columns('Sample Size 100', sample_props_100,
                     'Sample Size 1000', sample_props_1000).hist(bins=np.arange(0, 0.36, 0.01))
plt.scatter(level2_perfect_prop, 0, color='red', marker='^', s=60, zorder=3)
plt.title('Sample Proportions of Perfect Attempts on Level 2')
plt.xlabel('Sample Proportion')
plt.show()

---

### Task 2.4 📍

Compute the mean and standard deviation of each array of sample proportions, using `np.mean` and `np.std`. Assign:
* `mean_props_100` to the mean of `sample_props_100`
* `sd_props_100` to the standard deviation of `sample_props_100`
* `mean_props_1000` to the mean of `sample_props_1000`
* `sd_props_1000` to the standard deviation of `sample_props_1000`.

_Points:_ 4

In [ ]:
mean_props_100 = ...
sd_props_100 = ...
mean_props_1000 = ...
sd_props_1000 = ...

# Leave the following code to print out the calculated values.
print(f'For sample_props_100, the mean is approximately \
{mean_props_100:.4f} and the standard deviation is approximately {sd_props_100:.4f}.')
print(f'For sample_props_1000, the mean is approximately \
{mean_props_1000:.4f} and the standard deviation is approximately {sd_props_1000:.4f}.')

In [ ]:
grader.check("task_2_4")

---

### 🤔 Reflection

Consider the following questions. Your responses are not graded here, and you don't need to provide your responses in this notebook. However, your instructor may ask you to answer questions like these on the Midterm Exam, so it's a good idea to think them through or jot down notes now.

> 1. Using the above summary statistics from Task 2.4, what happened to the mean and standard deviation of the sample proportions when the sample size increased from 100 to 1,000, and why is that useful to a researcher?
2. Suppose you sampled 100 attempts on Level 2 and found that 25% were perfect. Based on the histogram, would that be surprising if the true proportion is about 16%?

---

## 🧪 Section 3: Testing a Claim

Suppose the article's authors had published an early result after only the first 500 players had finished Level 2, and that a reader wants to check their claim that **about 16% of players solve the lawn perfectly**. All the reader has is that early sample. For this section, treat the first 500 attempts on Level 2 as a random sample of all players who attempt Level 2.

Run the following code cell to create the table `early_sample`.

In [ ]:
early_sample = level2.take(np.arange(500))
early_sample.show(3)

---

### Hypotheses

The hypotheses you should use for this section are:

* **Null Hypothesis:** In the population of all attempts on Level 2, exactly 16\% are perfect. Any difference between 16% and the proportion in our sample is due to chance.
* **Alternative Hypothesis:** In the population of all attempts on Level 2, the proportion of perfect attempts is **not** 16%.

---

### Task 3.1 📍

Assign `n_early` to the number of attempts in `early_sample` and `observed_prop_early` to the **proportion** of those attempts that were perfect.

_Points:_ 2

In [ ]:
n_early = ...
observed_prop_early = ...

# Leave the following code to print out the calculated values.
print(f'Of the {n_early} early attempts, {observed_prop_early} were perfect.')

In [ ]:
grader.check("task_3_1")

---

### Developing a Test Statistic

The alternative says the proportion is **different** from 16%, whether higher or lower, so we need a test statistic where large values point toward the alternative in either direction. Use the test statistic defined as **the absolute value of the difference between the sample proportion and 16%.** 

Run the following code cell to save the claimed proportion.

In [ ]:
claimed_prop = 0.16

---

### Task 3.2 📍

Define a function called `compute_early_test_statistic` that takes a sample proportion as its only argument and returns the test statistic defined above, using `claimed_prop`.

Lastly, call your function on `observed_prop_early` and assign the result to `early_observed_stat`.

_Points:_ 3

In [ ]:
# Define the function
...

# Call the function
early_observed_stat = ...
early_observed_stat

In [ ]:
grader.check("task_3_2")

---

### Task 3.3 📍

Define a function called `simulate_one_early_stat` that takes no arguments and simulates **one** value of the test statistic assuming the null hypothesis is true. It should:

* Simulate a random sample of `n_early` attempts in which each attempt is perfect with probability `claimed_prop`.
* Compute the proportion of perfect attempts in the simulated sample.
* Return the test statistic for that simulated proportion.

Lastly, call your function once and assign the result to `one_simulated_early_stat`.

**Hint**: The `sample_proportions` function takes a sample size and an array of probabilities, and returns an array of proportions that fall in each category.

_Points:_ 3

In [ ]:
# Define the function
...

# Call the function
one_simulated_early_stat = ...
one_simulated_early_stat

In [ ]:
grader.check("task_3_3")

---

### Task 3.4 📍

Simulate the test statistic **5,000** times under the null hypothesis by calling `simulate_one_early_stat` inside a `for` loop, and store the 5,000 values in an array called `early_simulated_stats`.

_Points:_ 3

In [ ]:
early_simulated_stats = ...

...

In [ ]:
grader.check("task_3_4")

---

### Distribution of Test Statistics

Run the following code to generate a histogram of the simulated test statistics in `early_simulated_stats` along with the observed test statistic.

In [ ]:
Table().with_column('Simulated Test Statistic', early_simulated_stats).hist(bins=np.arange(0, 0.06, 0.004))
plt.scatter(early_observed_stat, 0, color='red', marker='^', s=60, zorder=3)
plt.title('Distribution of Test Statistics Under the Null')
plt.show()

---

### Task 3.5 📍

Compute the p-value for this hypothesis test and assign it to `early_p_value`.

_Points:_ 2

In [ ]:
early_p_value = ...
early_p_value

In [ ]:
grader.check("task_3_5")

---

### Task 3.6 📍

Which of the following is the best conclusion from this test? Assign `early_conclusion` to the number (1, 2, or 3) of your choice.

1. The data are consistent with the null hypothesis: a sample like this one is not surprising if 16% of all Level 2 attempts are perfect, so we do not have evidence against the claim.
2. The data favor the alternative hypothesis: the proportion of perfect attempts is not 16%.
3. The test proves that exactly 16% of all Level 2 attempts are perfect.

_Points:_ 2

In [ ]:
early_conclusion = ...
early_conclusion

In [ ]:
grader.check("task_3_6")

---

## 🔀 Section 4: A/B Testing

The article's central finding is that **when** players think matters more than **how much** they think. In this section you will compare groups of players using **A/B tests**, which use random shuffling to decide whether an observed difference between two groups could plausibly be due to chance alone.

To keep the simulations fast, we will use the first 1,500 attempts on Level 2, one for each of 1,500 different players. As in the last section, you can treat these as a random sample of Level 2 players. 

Run the following code cell to create the table `ab_sample`.

In [ ]:
ab_sample = level2.take(np.arange(1500))
ab_sample.show(3)

---

### Pausers vs. Rushers

Do players who stop to think **before their first move** end up with better routes? We will define two groups using `first_move_seconds`. A player is a **Pauser** if they waited at least 5 seconds before their first move, and a **Rusher** otherwise.

For the first part of this section, use the following hypotheses:

* **Null Hypothesis:** In the population, the distribution of optimality is the same for Pausers and Rushers. Any difference in our sample is due to chance.
* **Alternative Hypothesis:** In the population, Pausers have **higher** optimality, on average, than Rushers.

---

### Task 4.1 📍

Create a table called `ab_table` with three columns: `'platform'` and `'optimality'` from `ab_sample` (above), and a new column `'group'` that contains `'Pauser'` or `'Rusher'` for each player, following the definition above.

**Hint**: We've created a function `pauser_or_rusher` which could be helpful for you to use. 

_Points:_ 3

In [ ]:
def pauser_or_rusher(...):
    ...
        
ab_table = ...
ab_table.show(5)

In [ ]:
grader.check("task_4_1")

---

### Developing a Test Statistic

For the alternative hypothesis above, we need a statistic where larger values point toward the alternative. Use **the average optimality of the group that comes first alphabetically (Pausers) minus the average optimality of the group that comes second (Rushers)**. In the next tasks, you will write functions that work for **any** table with a numeric column and a column of two group labels, so you can reuse them later.

---

### Task 4.2 📍

Define a function called `difference_of_means` that takes three arguments: a table, the label of a numeric column, and the label of a column that contains two group labels. It should return the average of the numeric column for the group that comes **first alphabetically** minus the average for the group that comes second.

Lastly, call your function on `ab_table` to compute the observed test statistic comparing `'optimality'` between the two values in `'group'`, and assign it to `observed_pauser_diff`.

**Hint**: After grouping with `np.mean`, the rows of the output are sorted by group label.

_Points:_ 4

In [ ]:
# Define the function
...

# Call the function
observed_pauser_diff = ...
observed_pauser_diff

In [ ]:
grader.check("task_4_2")

---

### Task 4.3 📍

Define a function called `one_simulated_difference` that simulates **one** value of the test statistic under the null hypothesis. It should take the same three arguments as `difference_of_means` (a table, the numeric column label, and the group column label) and:

* Randomly shuffle the group labels, so the shuffled labels are unrelated to the numeric values.
* Return the difference of means computed with the shuffled labels.

Lastly, call your function on `ab_table` and assign the result to `one_pauser_simulated_diff`.

**Hint**: Sampling all the rows of a table **without replacement** shuffles them. Be careful not to change the original table.

_Points:_ 3

In [ ]:
# Define the function
...

# Call the function
one_pauser_simulated_diff = ...
one_pauser_simulated_diff

In [ ]:
grader.check("task_4_3")

---

### Task 4.4 📍

Simulate the test statistic **2,000** times under the null hypothesis by calling `one_simulated_difference` inside a `for` loop, and store the 2,000 values in an array called `pauser_simulated_diffs`. 

*This code cell may take up to a minute to run.*

_Points:_ 3

In [ ]:
pauser_simulated_diffs = ...

...

In [ ]:
grader.check("task_4_4")

---

### Distribution of Test Statistics

Run the following code to see the distribution of the simulated differences along with the observed difference.

In [ ]:
Table().with_column('Simulated Difference in Mean Optimality', pauser_simulated_diffs).hist(bins=np.arange(-4, 5, 0.25))
plt.scatter(observed_pauser_diff, 0, color='red', marker='^', s=60, zorder=3)
plt.title('Pausers vs. Rushers: Simulated Differences Under the Null')
plt.show()

---

### Task 4.5 📍

Compute the p-value for this A/B test and assign it to `pauser_p_value`.

_Points:_ 2

In [ ]:
pauser_p_value = ...
pauser_p_value

In [ ]:
grader.check("task_4_5")

---

### Task 4.6 📍

Which of the following is the best conclusion from this A/B test? Assign `pauser_conclusion` to the number (1, 2, or 3) of your choice.

1. The difference in mean optimality could easily be due to chance, so there is no evidence that Pausers and Rushers differ.
2. Pausers have higher optimality on average, and chance alone is very unlikely to explain the difference. However, because players were not randomly assigned to pause or rush, we cannot conclude that pausing itself causes higher optimality.
3. Pausing before the first move causes players to score higher on Level 2.

_Points:_ 2

In [ ]:
pauser_conclusion = ...
pauser_conclusion

In [ ]:
grader.check("task_4_6")

---

### 🤔 Reflection

Consider the following question. Your responses are not graded here, and you don't need to provide your answer in this notebook. However, your instructor may ask you to answer a question like this on the Midterm Exam, so it's a good idea to think it through or jot down notes now.

> What does the p-value you computed in Task 4.5 tell us? Refer to Pausers, Rushers, and shuffled labels in your answer.

---

### Desktop vs. Mobile Players

Now use the same tools for a different comparison. Desktop players steer the mower with a keyboard, while mobile players tap an on-screen keypad. Do players on the two platforms solve Level 2 differently? Here, you have no reason to expect a difference in a particular direction. 

For the rest of this section, use the following hypotheses:

* **Null Hypothesis:** In the population, the distribution of optimality is the same for desktop and mobile players. Any difference in our sample is due to chance.
* **Alternative Hypothesis:** In the population, the average optimality of desktop players is **different** from that of mobile players (in either direction).

Because the alternative is two-sided, use the **absolute value** of the difference in average optimality as the test statistic.

---

### Task 4.7 📍

Using your functions from earlier and the `'platform'` column of `ab_table`:

* Assign `observed_platform_stat` to the observed value of the test statistic (the absolute difference in mean optimality between desktop and mobile players).
* Simulate the test statistic **2,000** times under the null hypothesis and store the values in an array called `platform_simulated_stats`.

**Hint**: Use the `abs` function on the output of `difference_of_means` and `one_simulated_difference`.

_Points:_ 4

In [ ]:
observed_platform_stat = ...
platform_simulated_stats = ...

...

In [ ]:
grader.check("task_4_7")

---

### Task 4.8 📍

Compute the p-value for the desktop vs. mobile A/B test and assign it to `platform_p_value`.

_Points:_ 2

In [ ]:
platform_p_value = ...
platform_p_value

In [ ]:
grader.check("task_4_8")

---

### Task 4.09 📍

Which of the following is the best conclusion from this A/B test? Assign `platform_conclusion` to the number (1, 2, or 3) of your choice.

1. There is strong evidence that desktop and mobile players differ in average optimality.
2. The observed difference is well within what random shuffling produces, so the data are consistent with no difference in the population between desktop and mobile players.
3. The test proves that the platform has no effect on how well anyone plays.

_Points:_ 2

In [ ]:
platform_conclusion = ...
platform_conclusion

In [ ]:
grader.check("task_4_09")

---

### 🤔 Reflection

Consider the following question. Your responses are not graded here, and you don't need to provide your answer in this notebook. However, your instructor may ask you to answer a question like this on the Midterm Exam, so it's a good idea to think it through or jot down notes now.

> The two A/B tests in this section gave very different p-values. Why do you think this is? In your reflection, compare `observed_pauser_diff` with `observed_platform_stat`, and compare each to the typical size of the simulated differences (for example, by looking at the two histograms or at the standard deviations of the simulated statistics).

---

## 🎉 Wrapping Up

Great work making it through this project! You've just used many of the skills and concepts from the course, including table manipulation, visualization, functions, random sampling, hypothesis testing, and A/B testing, to explore how real people plan routes in a puzzle. Step back and think about how far you've come since day one of this course!

One last thing to keep in mind: the players in this dataset chose to play an online game, so even a very small p-value does not tell us how the results would look for people in general. Good statistical reasoning includes asking who is in the data and how they got there.

---

## Final Task 📍

<!-- BEGIN QUESTION -->

You're almost done! Run the cell below to check that you passed all of the non-hidden auto-graded tasks, and then follow the instructions below to submit your assignment to Canvas.

_Points:_ 0

In [ ]:
grader.check_all()

<!-- END QUESTION -->

---

## 🏁 Submit Your Assignment to Canvas

1. **Review the Rubric:** View the rubric on the Canvas assignment page to understand the scoring criteria.  
2. **Review Manually Graded Tasks:** Verify that you have responded to all manually graded tasks marked with 📍🔎.  
3. **Review Auto-Graded Tasks:** Inspect the output from the `grader.check_all()` command above. Ensure you have passed all available tests for the auto-graded tasks in this notebook marked with 📍.  
   - This command executes all auto-grader tests sequentially.  
   - Note that Homework and Project assignments include hidden auto-grader tests that are not visible to you. Passing all visible tests does **not** guarantee that your solution is fully correct.  
4. **Save Your Work:** In the notebook toolbar, go to `File → Save default` to save your work and create a checkpoint.  
5. **Download the Notebook:** In the notebook toolbar, go to `File → Download IPYNB` to download the notebook (`.ipynb`) file.  
6. **Upload to Canvas:** Upload the downloaded `.ipynb` file to the appropriate Canvas assignment page.

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>